# Evaluación cuantitativa del sistema InfiniteRecs## TFG — Ingeniería del Software 2025-26Este notebook evalúa el modelo Two-Tower desde múltiples ángulos,reutilizando exactamente la misma lógica de inferencia que la web (`recommender.py`).**Secciones:**1. Escenario A vs B (embedding aprendido vs cold-start)2. Análisis por género de película3. Análisis por popularidad4. Análisis por actividad del usuario5. Validación con watchlist

## 0. Setup

In [ ]:
import sys, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import zipfile, io
from collections import defaultdict, Counter
from pathlib import Path
from scipy import stats

# Importar desde el backend de la web (misma lógica de inferencia)
sys.path.insert(0, os.path.abspath("../backend"))
from recommender import Recommender
from zip_processor import extract_username_from_filename, load_seen_titles_from_zip_bytes

# Configuración (debe coincidir con el notebook de entrenamiento)
POSITIVE_THRESHOLD = 0.6
VAL_SPLIT = 0.2
MIN_POSITIVES_PER_USER = 5
TOP_K = 10
SEED = 42

# Rutas
BACKEND_DIR = Path("../backend")
ZIPS_DIR = Path("zips")
FIGURES_DIR = Path("figures")
FIGURES_DIR.mkdir(exist_ok=True)

# Estilo de gráficas
plt.rcParams.update({
    'figure.facecolor': '#0a0a0a',
    'axes.facecolor': '#0a0a0a',
    'axes.edgecolor': '#555',
    'axes.labelcolor': '#ccc',
    'text.color': '#ccc',
    'xtick.color': '#aaa',
    'ytick.color': '#aaa',
    'grid.color': '#333',
    'grid.alpha': 0.5,
    'font.size': 11,
})

print("Setup OK")


## 0.1 Cargar modelo (misma clase que usa la web)

In [ ]:
rec = Recommender(
    checkpoint_path=str(BACKEND_DIR / "checkpoint.pt"),
    movies_csv_path=str(BACKEND_DIR / "movies.csv"),
)
print(f"Modelo: v{rec.version}")
print(f"Usuarios entrenados: {rec.n_users}")
print(f"Películas en catálogo: {rec.n_movies}")
print(f"Métricas del checkpoint: {rec.metrics}")
print(f"Usuarios conocidos: {sorted(rec.user2id.keys())}")


## 0.2 Cargar ZIPs y construir split temporalLeemos los ZIPs de Letterboxd, extraemos ratings con fechas, hacemos elmismo split temporal 80/20 que usó el notebook de entrenamiento, ypreparamos las estructuras de datos para la evaluación.

In [ ]:
zip_paths = sorted(ZIPS_DIR.glob("*.zip"))
print(f"ZIPs encontrados: {len(zip_paths)}\n")

user_data = {}

for zp in zip_paths:
    username = extract_username_from_filename(zp.name)
    if not username:
        print(f"  SKIP: {zp.name} (formato no reconocido)")
        continue

    zip_bytes = zp.read_bytes()

    # --- Ratings con fechas (necesarias para el split temporal) ---
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
        rat_files = [n for n in z.namelist() if n.endswith("ratings.csv")]
        if not rat_files:
            print(f"  SKIP: {username} (sin ratings.csv)")
            continue
        with z.open(rat_files[0]) as f:
            ratings_raw = pd.read_csv(f)

    ratings_raw = ratings_raw.dropna(subset=["Name", "Rating"]).copy()
    ratings_raw["title_normalized"] = ratings_raw["Name"].str.lower()
    ratings_raw["rating_norm"] = (ratings_raw["Rating"] - 0.5) / 4.5
    if "Date" in ratings_raw.columns:
        ratings_raw["date"] = pd.to_datetime(ratings_raw["Date"], errors="coerce")

    # --- Watchlist ---
    watchlist = set()
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
        wl_files = [n for n in z.namelist() if n.endswith("watchlist.csv")]
        if wl_files:
            with z.open(wl_files[0]) as f:
                wl_df = pd.read_csv(f)
            if "Name" in wl_df.columns:
                watchlist = set(wl_df["Name"].dropna().str.lower())

    # --- Seen (ratings + watched) para exclusión ---
    seen = load_seen_titles_from_zip_bytes(zip_bytes)
    seen |= set(ratings_raw["title_normalized"])

    # --- Filtrar a películas en el catálogo del modelo ---
    in_catalog = ratings_raw[ratings_raw["title_normalized"].isin(rec.movie2id)].copy()

    user_data[username] = {
        "ratings_in_catalog": in_catalog,
        "watchlist": watchlist,
        "seen_titles": seen,
        "n_total_ratings": len(ratings_raw),
        "n_in_catalog": len(in_catalog),
        "is_known": rec.is_known_user(username),
    }
    scenario = "A (conocido)" if rec.is_known_user(username) else "B (cold-start)"
    print(f"  {username:20s} | {scenario} | {len(in_catalog):4d} en catálogo | {len(watchlist):4d} watchlist")

print(f"\nTotal: {len(user_data)} usuarios cargados")


In [ ]:
# Split temporal: mismo protocolo que el entrenamiento
train_info = {}
val_positives = {}

for username, ud in user_data.items():
    df = ud["ratings_in_catalog"].copy()
    positives = df[df["rating_norm"] > POSITIVE_THRESHOLD].copy()

    if len(positives) < MIN_POSITIVES_PER_USER:
        print(f"  SKIP {username}: solo {len(positives)} positivos (mín {MIN_POSITIVES_PER_USER})")
        continue

    # Ordenar por fecha y tomar el último 20% como val
    if "date" in positives.columns and positives["date"].notna().any():
        positives = positives.sort_values("date")

    n_val = max(1, int(len(positives) * VAL_SPLIT))
    train_pos = positives.iloc[:-n_val]
    val_pos = positives.iloc[-n_val:]

    train_info[username] = {
        "positive_titles": set(train_pos["title_normalized"]),
        "all_rated_titles": set(df["title_normalized"]),
        "ratings_df": df[["title_normalized", "rating_norm"]].copy(),
        "n_train_pos": len(train_pos),
    }
    val_positives[username] = set(val_pos["title_normalized"])

    print(f"  {username:20s} | train: {len(train_pos):3d} pos | val: {len(val_pos):3d} pos")

print(f"\nUsuarios evaluables: {len(val_positives)}")


## 0.3 Funciones de evaluación

In [ ]:
def evaluate_user(rec, user_embedding, seen_titles, val_pos_titles, top_k=TOP_K):
    """Evalúa P@K y R@K para un usuario.

    Excluye las películas vistas EXCEPTO las val positives, para que estas
    puedan aparecer en el ranking. Esto es idéntico al protocolo del
    notebook de entrenamiento.
    """
    exclude = seen_titles - val_pos_titles
    results = rec.recommend(user_embedding, exclude, top_k=top_k)
    rec_titles = {r["title_normalized"] for r in results}
    hits = rec_titles & val_pos_titles
    return {
        "precision": len(hits) / top_k,
        "recall": len(hits) / len(val_pos_titles) if val_pos_titles else 0,
        "n_hits": len(hits),
        "n_val": len(val_pos_titles),
        "hit_titles": hits,
        "recommendations": results,
        "rec_titles": rec_titles,
    }


def get_genres_for_title(rec, title_normalized):
    """Devuelve la lista de géneros TMDB de una película."""
    if title_normalized in rec.movies_by_title.index:
        genres_str = rec.movies_by_title.loc[title_normalized].get("tmdb_genres", "")
        if genres_str and str(genres_str) != "nan":
            return [g.strip() for g in str(genres_str).split("|") if g.strip()]
    return []

print("Funciones de evaluación definidas")


## 1. Escenario A vs Escenario B**Escenario A**: el usuario está entre los 20 entrenados → se usa su embedding aprendido.**Escenario B**: se simula cold-start → se construye el embedding con fold-in(ponderación cuadrática + anti-embedding + gradient descent).Para cada usuario entrenado, evaluamos ambos escenarios sobre el mismo val set.Esto cuantifica cuánto valor aporta el embedding aprendido frente al cold-start.

In [ ]:
results_a = {}
results_b = {}

for username in val_positives:
    val_pos = val_positives[username]
    seen = user_data[username]["seen_titles"]
    is_known = user_data[username]["is_known"]

    # --- Escenario A (solo si el usuario está entrenado) ---
    if is_known:
        emb_a = rec.get_user_embedding_known(username)
        results_a[username] = evaluate_user(rec, emb_a, seen, val_pos)

    # --- Escenario B (cold-start con ratings de TRAIN solamente) ---
    # Usamos solo train ratings para que la comparación con A sea justa:
    # ambos escenarios ven la misma información.
    train_ratings_df = train_info[username]["ratings_df"]
    # Filtrar a solo las películas que estaban en train (no val)
    train_titles = train_info[username]["all_rated_titles"] - val_pos
    df_for_coldstart = train_ratings_df[train_ratings_df["title_normalized"].isin(train_titles)].copy()

    try:
        cs_result = rec.build_coldstart_embedding(
            df_for_coldstart,
            use_quadratic=True,
            use_anti=True,
            use_finetuning=True,
        )
        emb_b = cs_result["embedding"]
        results_b[username] = evaluate_user(rec, emb_b, seen, val_pos)
    except ValueError as e:
        print(f"  {username}: cold-start falló ({e})")
        results_b[username] = {"precision": 0, "recall": 0, "n_hits": 0, "n_val": len(val_pos)}

print("Evaluación A/B completada")


In [ ]:
# Tabla comparativa
rows = []
for username in sorted(val_positives.keys()):
    p_a = results_a[username]["precision"] if username in results_a else None
    p_b = results_b[username]["precision"] if username in results_b else None
    r_a = results_a[username]["recall"] if username in results_a else None
    r_b = results_b[username]["recall"] if username in results_b else None
    n_val = val_positives[username]

    rows.append({
        "username": username,
        "n_val_positives": len(n_val),
        "P@10 (A)": p_a,
        "P@10 (B)": p_b,
        "Δ P@10": (p_a - p_b) if (p_a is not None and p_b is not None) else None,
        "R@10 (A)": r_a,
        "R@10 (B)": r_b,
    })

df_ab = pd.DataFrame(rows)

# Medias
mean_a = df_ab["P@10 (A)"].dropna().mean()
mean_b = df_ab["P@10 (B)"].dropna().mean()
print(f"Media P@10 Escenario A: {mean_a:.4f}")
print(f"Media P@10 Escenario B: {mean_b:.4f}")
print(f"Diferencia media: {mean_a - mean_b:+.4f} ({(mean_a - mean_b) / mean_a * 100:+.1f}%)\n")

print(df_ab.to_string(index=False, float_format="%.4f"))


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

users = df_ab.sort_values("Δ P@10", ascending=False)["username"]
x = np.arange(len(users))
w = 0.35

a_vals = [df_ab.loc[df_ab["username"] == u, "P@10 (A)"].values[0] or 0 for u in users]
b_vals = [df_ab.loc[df_ab["username"] == u, "P@10 (B)"].values[0] or 0 for u in users]

ax.bar(x - w/2, a_vals, w, label="Escenario A (embedding aprendido)", color="#FF8000")
ax.bar(x + w/2, b_vals, w, label="Escenario B (cold-start)", color="#40BCF4")
ax.set_xticks(x)
ax.set_xticklabels(users, rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Precision@10")
ax.set_title("Escenario A vs B por usuario")
ax.legend(loc="upper right")
ax.grid(axis="y")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "scenario_ab.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Guardado en {FIGURES_DIR / 'scenario_ab.png'}")


## 2. Análisis por género¿El modelo recomienda mejor ciertos géneros que otros? Para cada género TMDB,calculamos el hit rate: de los positivos de val que tienen ese género,¿cuántos aparecen en el top-10 del modelo?

In [ ]:
# Para cada usuario, por cada hit y miss en val, anotar sus géneros
genre_hits = Counter()
genre_total = Counter()

for username in results_a:
    eval_result = results_a[username]
    val_pos = val_positives[username]

    # Géneros de las películas en val
    for title in val_pos:
        genres = get_genres_for_title(rec, title)
        for g in genres:
            genre_total[g] += 1
            if title in eval_result["hit_titles"]:
                genre_hits[g] += 1

# Hit rate por género
genre_stats = []
for genre in sorted(genre_total.keys()):
    total = genre_total[genre]
    hits = genre_hits.get(genre, 0)
    if total >= 3:  # mínimo 3 películas para que sea representativo
        genre_stats.append({
            "genre": genre,
            "n_val": total,
            "hits": hits,
            "hit_rate": hits / total,
        })

df_genre = pd.DataFrame(genre_stats).sort_values("hit_rate", ascending=True)
print(df_genre.to_string(index=False, float_format="%.3f"))


In [ ]:
fig, ax = plt.subplots(figsize=(10, max(5, len(df_genre) * 0.35)))

colors = ["#00E054" if hr > df_genre["hit_rate"].median() else "#40BCF4"
          for hr in df_genre["hit_rate"]]

ax.barh(df_genre["genre"], df_genre["hit_rate"], color=colors)
ax.set_xlabel("Hit Rate (aciertos / total en val)")
ax.set_title("Hit rate por género TMDB")

# Anotar el nº de películas en val para cada género
for i, (_, row) in enumerate(df_genre.iterrows()):
    ax.text(row["hit_rate"] + 0.005, i, f'n={row["n_val"]:.0f}',
            va="center", fontsize=8, color="#888")

ax.grid(axis="x")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "genre_hitrate.png", dpi=150, bbox_inches="tight")
plt.show()


## 3. Análisis por popularidad¿El modelo favorece películas populares? Clasificamos cada película delcatálogo por cuántos usuarios del dataset la han rateado, y vemos cómose distribuyen los aciertos entre franjas de popularidad.

In [ ]:
# Contar popularidad de cada película (cuántos usuarios la han rateado)
movie_popularity = Counter()
for username in train_info:
    for title in train_info[username]["all_rated_titles"]:
        movie_popularity[title] += 1

def pop_category(title):
    n = movie_popularity.get(title, 0)
    if n >= 10: return "Popular (≥10)"
    if n >= 3: return "Media (3-9)"
    return "Nicho (1-2)"

# Para cada hit en escenario A, clasificar por popularidad
pop_hits = Counter()
pop_total = Counter()

for username in results_a:
    val_pos = val_positives[username]
    hit_titles = results_a[username]["hit_titles"]
    for title in val_pos:
        cat = pop_category(title)
        pop_total[cat] += 1
        if title in hit_titles:
            pop_hits[cat] += 1

pop_stats = []
for cat in ["Popular (≥10)", "Media (3-9)", "Nicho (1-2)"]:
    total = pop_total.get(cat, 0)
    hits = pop_hits.get(cat, 0)
    pop_stats.append({
        "categoría": cat,
        "n_val": total,
        "hits": hits,
        "hit_rate": hits / total if total > 0 else 0,
    })

df_pop = pd.DataFrame(pop_stats)
print(df_pop.to_string(index=False, float_format="%.3f"))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Barplot de hit rate
colors_pop = ["#FF8000", "#40BCF4", "#00E054"]
axes[0].bar(df_pop["categoría"], df_pop["hit_rate"], color=colors_pop)
axes[0].set_ylabel("Hit Rate")
axes[0].set_title("Hit rate por franja de popularidad")
axes[0].grid(axis="y")

# Pie de distribución de hits
hit_vals = df_pop["hits"].values
if hit_vals.sum() > 0:
    axes[1].pie(hit_vals, labels=df_pop["categoría"], autopct="%1.0f%%",
                colors=colors_pop, textprops={"color": "#ccc"})
    axes[1].set_title("Distribución de aciertos")
else:
    axes[1].text(0.5, 0.5, "Sin hits", ha="center", va="center")
    axes[1].set_title("Distribución de aciertos")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "popularity_analysis.png", dpi=150, bbox_inches="tight")
plt.show()


## 4. Análisis por actividad del usuario¿Los usuarios con más ratings reciben mejores recomendaciones?Scatter plot: eje X = nº de ratings en train, eje Y = P@10.Calculamos la correlación de Pearson.

In [ ]:
user_scatter = []
for username in results_a:
    n_ratings = train_info[username]["n_train_pos"]
    p10 = results_a[username]["precision"]
    user_scatter.append({"username": username, "n_train_pos": n_ratings, "P@10": p10})

df_scatter = pd.DataFrame(user_scatter)

# Correlación de Pearson
r, p_value = stats.pearsonr(df_scatter["n_train_pos"], df_scatter["P@10"])
print(f"Correlación de Pearson: r = {r:.3f}, p-value = {p_value:.4f}")
print(f"Interpretación: {'Positiva' if r > 0 else 'Negativa'} {'significativa' if p_value < 0.05 else 'no significativa'}")
print()
print(df_scatter.to_string(index=False, float_format="%.4f"))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(df_scatter["n_train_pos"], df_scatter["P@10"],
           color="#FF8000", s=80, edgecolors="#444", zorder=3)

# Línea de tendencia
z = np.polyfit(df_scatter["n_train_pos"], df_scatter["P@10"], 1)
p = np.poly1d(z)
x_range = np.linspace(df_scatter["n_train_pos"].min(), df_scatter["n_train_pos"].max(), 100)
ax.plot(x_range, p(x_range), "--", color="#00E054", linewidth=1.5, alpha=0.7,
        label=f"Tendencia (r={r:.2f})")

# Etiquetas de usuario
for _, row in df_scatter.iterrows():
    ax.annotate(row["username"], (row["n_train_pos"], row["P@10"]),
                textcoords="offset points", xytext=(5, 5), fontsize=7, color="#999")

ax.set_xlabel("Nº de positivos en train")
ax.set_ylabel("Precision@10")
ax.set_title("P@10 vs actividad del usuario")
ax.legend()
ax.grid(True)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "user_activity_scatter.png", dpi=150, bbox_inches="tight")
plt.show()


## 5. Validación con watchlistLa watchlist de Letterboxd contiene películas que el usuario **quiere verpero aún no ha visto**. Si el modelo las recomienda, es una validaciónindependiente del val set: no predecimos el pasado (películas ya vistas)sino la **intención futura** del usuario.Métrica: **Watchlist Hit Rate** = de las top-K recomendaciones (excluyendotodo lo visto), ¿cuántas están en la watchlist del usuario?

In [ ]:
wl_results = []

for username in results_a:
    watchlist = user_data[username]["watchlist"]
    watchlist_in_catalog = watchlist & set(rec.movie2id.keys())

    if not watchlist_in_catalog:
        continue

    # Recomendaciones del escenario A (excluyendo TODO lo visto, incluido val)
    # Para watchlist usamos exclusión COMPLETA, como haría la web en producción.
    emb = rec.get_user_embedding_known(username)
    seen = user_data[username]["seen_titles"]
    recs = rec.recommend(emb, seen, top_k=TOP_K)
    rec_titles = {r["title_normalized"] for r in recs}

    wl_hits = rec_titles & watchlist_in_catalog
    wl_hit_rate = len(wl_hits) / TOP_K

    wl_results.append({
        "username": username,
        "watchlist_size": len(watchlist_in_catalog),
        "wl_hits": len(wl_hits),
        "wl_hit_rate": wl_hit_rate,
        "hit_titles": wl_hits,
    })

    if wl_hits:
        print(f"  {username}: {len(wl_hits)}/{TOP_K} recs en watchlist → {list(wl_hits)}")

df_wl = pd.DataFrame(wl_results)
if len(df_wl) > 0:
    print(f"\nMedia Watchlist Hit Rate: {df_wl['wl_hit_rate'].mean():.4f}")
    print(f"Usuarios con al menos 1 hit en watchlist: {(df_wl['wl_hits'] > 0).sum()}/{len(df_wl)}")
    print()
    print(df_wl.to_string(index=False, float_format="%.4f"))
else:
    print("Ningún usuario tiene watchlist en el catálogo.")


In [ ]:
if len(df_wl) > 0:
    fig, ax = plt.subplots(figsize=(12, 5))

    df_wl_sorted = df_wl.sort_values("wl_hit_rate", ascending=False)
    colors_wl = ["#00E054" if hr > 0 else "#333" for hr in df_wl_sorted["wl_hit_rate"]]

    ax.bar(df_wl_sorted["username"], df_wl_sorted["wl_hit_rate"], color=colors_wl)
    ax.set_ylabel("Watchlist Hit Rate")
    ax.set_title("¿Cuántas recomendaciones coinciden con la watchlist del usuario?")
    ax.set_xticklabels(df_wl_sorted["username"], rotation=45, ha="right", fontsize=8)
    ax.axhline(y=df_wl["wl_hit_rate"].mean(), color="#FF8000", linestyle="--",
               label=f'Media: {df_wl["wl_hit_rate"].mean():.3f}')
    ax.legend()
    ax.grid(axis="y")

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "watchlist_validation.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No se pudo generar gráfica de watchlist (sin datos).")


## 6. Resumen de figuras generadasLas figuras se han guardado en `evaluation/figures/`:- `scenario_ab.png` — Comparativa Escenario A vs B por usuario- `genre_hitrate.png` — Hit rate por género TMDB- `popularity_analysis.png` — Análisis por franja de popularidad- `user_activity_scatter.png` — P@10 vs actividad del usuario (scatter)- `watchlist_validation.png` — Validación con watchlistTodas listas para incluir directamente en la memoria (LaTeX: `\includegraphics`).